# Ribeiro e Souza — Leitor de Processos
### Do PDF ao JSON: como o MVP descobre se a dívida do Estado já foi paga

**Roteiro**

1. O problema e os dados
2. Arquitetura: os arquivos do projeto
3. Processos de demonstração *(fictícios)*
4. Passeio arquivo por arquivo: `constantes.py` → `erros.py` → `classes.py` → `funcoes.py`
5. O pipeline passo a passo, de um PDF até o **JSON**
6. `main.py`: tudo junto, pela linha de comando
7. Por que confiar no resultado
8. Limitações e próximos passos

> 🔒 **Todos os dados exibidos aqui são fictícios.** Os processos de demonstração são gerados dentro do próprio notebook, para não expor dados pessoais de clientes (CPF, RG, endereço) nem os PDFs reais do escritório.

In [1]:
import contextlib, html, inspect, json, os, re, subprocess, sys, tempfile, textwrap
from pathlib import Path
from unittest import mock

import pandas as pd
import pymupdf
from IPython.display import HTML, Markdown, display

# Este notebook fica em labs/, mas o projeto roda a partir da raiz (onde está a pasta app/).
RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "app").is_dir():
    RAIZ_PROJETO = RAIZ_PROJETO.parent
os.chdir(RAIZ_PROJETO)
sys.path.insert(0, str(RAIZ_PROJETO))

import app.funcoes as funcoes
from app import *                      # tudo o que o projeto exporta (funções, classes, erros, constantes)
from app import constantes as const

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)
print("Projeto em:", RAIZ_PROJETO)

Projeto em: /Users/nemo/nautilus/services/ribeiro_e_souza


In [2]:
# ---- Funções auxiliares de APRESENTAÇÃO (não fazem parte do projeto) ----

def mostrar_codigo(*objetos):
    """Mostra o código-fonte de funções/classes do projeto, com realce de sintaxe."""
    fontes = "\n\n".join(inspect.getsource(o) for o in objetos)
    display(Markdown(f"```python\n{fontes}\n```"))


def mostrar_arquivo(caminho):
    display(Markdown(f"```python\n{Path(caminho).read_text(encoding='utf-8')}\n```"))


def arvore(pasta=None, prefixo=""):
    """Árvore de arquivos. input/ e output/ não têm o conteúdo listado (nomes de clientes)."""
    pasta = pasta or RAIZ_PROJETO
    ignorar = {".git", ".venv", "__pycache__", ".ipynb_checkpoints", ".DS_Store"}
    itens = sorted((p for p in pasta.iterdir() if p.name not in ignorar), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(itens):
        ultimo = i == len(itens) - 1
        conector = "└── " if ultimo else "├── "
        if p.is_dir() and p.name in {"input", "output"}:
            n = sum(1 for f in p.iterdir() if f.name != ".gitkeep")
            print(f"{prefixo}{conector}{p.name}/    ({n} arquivo(s); conteúdo não listado: dados pessoais)")
        elif p.is_dir():
            print(f"{prefixo}{conector}{p.name}/")
            arvore(p, prefixo + ("    " if ultimo else "│   "))
        else:
            print(f"{prefixo}{conector}{p.name}")


def destacar(texto, termos):
    """Pinta no texto o que cada regex do dicionário encontrou."""
    achados = []
    for nome, padrao in termos.items():
        for m in re.finditer(padrao, texto, re.IGNORECASE):
            achados.append((m.start(), m.end(), nome))
    achados.sort()
    partes, pos = [], 0
    for ini, fim, nome in achados:
        if ini < pos:
            continue
        partes.append(html.escape(texto[pos:ini]))
        partes.append(f"<mark>{html.escape(texto[ini:fim])}</mark> <sub>[{nome}]</sub>")
        pos = fim
    partes.append(html.escape(texto[pos:]))
    display(HTML("<p style='font-size:1.05em;line-height:1.6'>" + "".join(partes) + "</p>"))


ROTULOS_ESCADA = [
    ("SENTENCA", "Sentença"),
    ("TRANSITO_EM_JULGADO", "Trânsito<br>em julgado"),
    ("CUMPRIMENTO_DE_SENTENCA", "Cumprimento<br>de sentença"),
    ("REQUISICAO", "Requisição<br>(RPV/OPV/precatório)"),
    ("PAGAMENTO_DISPONIBILIZADO", "Pagamento<br>disponibilizado"),
    ("QUITADO", "Quitado"),
]
COR_FASE = {"PAGAMENTO_DISPONIBILIZADO": "#1d4ed8", "QUITADO": "#15803d"}  # demais: âmbar


def desenhar_escada(fase_atual=None, legenda=""):
    """Desenha a escada de fases; as fases já alcançadas ficam coloridas e a atual leva uma borda."""
    atual = const.ORDEM_FASE.get(fase_atual, -1)
    caixas = []
    for codigo, rotulo in ROTULOS_ESCADA:
        alcancada = const.ORDEM_FASE[codigo] <= atual
        ativa = codigo == fase_atual
        fundo = COR_FASE.get(codigo, "#b45309") if alcancada else "#475569"
        borda = "#facc15" if ativa else "transparent"
        seta = "▶ " if ativa else ""
        caixas.append(
            f"<div style='background:{fundo};color:#fff;border:3px solid {borda};border-radius:8px;"
            f"padding:8px 10px;min-width:112px;text-align:center;font-size:13px;line-height:1.25'>{seta}{rotulo}</div>"
        )
    separador = "<div style='align-self:center;opacity:.6'>›</div>"
    rodape = f"<div style='margin:4px 0 8px'>{legenda}</div>" if legenda else ""
    display(HTML("<div style='display:flex;flex-wrap:wrap;gap:6px;margin:8px 0'>" + separador.join(caixas) + "</div>" + rodape))

---
## 1. O problema e os dados

**O problema.** Quando uma pessoa ganha uma ação contra o Estado (por exemplo, contra o INSS), o valor é pago por **RPV** (requisição de pequeno valor) ou **precatório**. O escritório quer comprar esse crédito, e antes precisa saber, rápido:

- Quem são as partes, o advogado e o juiz?
- Em que fase o crédito está?
- **A dívida já foi paga/liquidada?** (se já foi, não há o que comprar)

Hoje isso exige ler à mão autos de **70 a 300+ páginas**. O MVP lê o PDF e responde com **evidência: a página e o trecho** onde achou cada informação.

**A "escada" de fases do crédito.** O sistema classifica o processo na fase mais avançada que conseguir comprovar:

In [3]:
desenhar_escada(None, "Fases possíveis (nenhuma alcançada ainda). <b>Pergunta do escritório:</b> o processo já chegou em <i>Pagamento disponibilizado</i> ou <i>Quitado</i>?")

**Os dados.** A pesquisa inicial apontou o *Diário da Justiça*, mas os PDFs entregues pelo escritório são **autos completos do processo**, em dois sistemas diferentes:

| Sistema | Tribunal | Como se parece | O que o parser usa |
|---|---|---|---|
| **PJe** | TRF1 (Juizados Especiais Federais) | Capa estruturada + rodapé em toda página (`Num. <id> - Pág. N`) | Capa (partes, advogados); quem **assina a sentença** = juiz |
| **e-SAJ** | TJSP | Sem capa; petição já na primeira página | `Juiz(a) de Direito: Dr(a). ...` e rótulos `Exequente:` |

Os 3 PDFs reais testados (1 do TJSP e 2 do TRF1, de 72 a 306 páginas) **não são exibidos** aqui por conterem dados pessoais. O formato do Diário da Justiça (DJe/DJEN) ainda **não foi testado**.

**O número do processo (padrão CNJ)** já diz o tribunal: `NNNNNNN-DD.AAAA.J.TR.OOOO`, onde `J` é o ramo da Justiça (4 = federal, 8 = estadual) e `TR` o tribunal.

In [4]:
for numero in ["1234567-89.2025.4.01.3500", "0000001-23.2020.8.26.0100", "5000001-00.2024.8.09.0051"]:
    print(f"{numero}  ->  {identificar_tribunal(numero)}")

1234567-89.2025.4.01.3500  ->  TRF1
0000001-23.2020.8.26.0100  ->  TJSP
5000001-00.2024.8.09.0051  ->  TJGO


---
## 2. Arquitetura: os arquivos do projeto

In [5]:
arvore()

├── app/
│   ├── __init__.py
│   ├── classes.py
│   ├── constantes.py
│   ├── erros.py
│   └── funcoes.py
├── input/    (3 arquivo(s); conteúdo não listado: dados pessoais)
├── labs/
│   └── apresentacao.ipynb
├── output/    (4 arquivo(s); conteúdo não listado: dados pessoais)
├── tests/
│   ├── __init__.py
│   └── test_funcoes.py
├── .gitignore
├── README.md
├── environment.yml
├── main.py
└── requirements.txt


| Arquivo | Papel |
|---|---|
| `app/constantes.py` | ⭐ **Regras de negócio**: o dicionário de termos, os regex, as fases, os parâmetros |
| `app/erros.py` | Erros próprios (PDF não encontrado, protegido, corrompido) |
| `app/classes.py` | Estruturas de dados: `Pagina`, `Evidencia`, `ResultadoProcesso` |
| `app/funcoes.py` | O pipeline: ler → extrair → buscar → classificar → exportar |
| `main.py` | Linha de comando que junta tudo |
| `tests/` | 24 testes automatizados com textos sintéticos |
| `labs/` | Este notebook |

**O fluxo de dados:**

```
PDF ─► ler_pdf ─► identificar_documentos ─► extrair_metadados ─► buscar_termos ─► classificar_fase ─► exportar_json / csv
       texto       (PJe) tipo e assinante    nº, tribunal,        dicionário +      fase mais alta        resultado final
       por página  de cada página            partes, OAB, juiz    regex → evidências com evidência       (com as provas)
```

---
## 3. Processos de demonstração *(fictícios)*

Para a apresentação, o notebook **gera dois PDFs fictícios** no formato do PJe (capa, petição, sentença, certidão, decisão, ofício requisitório), com as mesmas "armadilhas" encontradas nos autos reais:

É o **mesmo processo fictício em duas datas**:

- **Processo A** (PDF de 15/10/2026): RPV expedida, **ainda não paga**. Contém texto-padrão no futuro (*"após o pagamento, será intimada para o levantamento…"*), uma manifestação da parte alegando quitação, e uma página escaneada.
- **Processo B** (PDF de 04/11/2026): o mesmo processo semanas depois, com certidão de depósito e decisão **extinguindo a execução pelo pagamento**.

*(A célula abaixo é "bastidores" e pode ficar recolhida na apresentação.)*

In [6]:
AUTOR = "MARIA APARECIDA DE DEMONSTRACAO"
ADVOGADO = "JOAO ADVOGADO EXEMPLO"
JUIZ = "JOSE JUIZ FEDERAL EXEMPLO"
SERVIDOR = "SERVIDOR EXEMPLO DA SILVA"
NUMERO_DEMO = "1234567-89.2025.4.01.3500"


def _escrever(pagina, linhas, y=56, tamanho=9.5, passo=13):
    for linha in linhas:
        for pedaco in (textwrap.wrap(linha, 92) or [""]):
            pagina.insert_text((46, y), pedaco, fontsize=tamanho)
            y += passo


def _pagina_documento(doc, doc_id, tipo, assinante, data, corpo):
    """Uma página no padrão PJe: cabeçalho 'Documento id', corpo e rodapé de assinatura."""
    pagina = doc.new_page()
    _escrever(pagina, [f"Documento id {doc_id} - {tipo}", ""] + corpo)
    pagina.insert_text((46, 790), f"Assinado eletronicamente por: {assinante} - {data}", fontsize=8)
    pagina.insert_text((46, 802), f"Num. {doc_id} - Pág. 1", fontsize=8)


def criar_pdf_demo(destino, quitado):
    doc = pymupdf.open()

    # --- capa (com a linha do tempo dos documentos) ---
    linha_do_tempo = [
        "10/03/2025  Petição inicial                      Polo ativo",
        "10/03/2025  Carteira de identidade (escaneada)   Polo ativo",
        "20/08/2025  Sentença Tipo A                      Interno",
        "20/09/2025  Certidão de Trânsito em Julgado      Interno",
        "05/10/2026  Decisão                              Interno",
        "12/10/2026  Ofício Requisitório de RPV - Requisição de Pequeno Valor     Interno",
        "14/10/2026  Manifestação                         Polo ativo",
    ]
    if quitado:
        linha_do_tempo += ["20/10/2026  Certidão                          Interno", "28/10/2026  Decisão                           Interno"]
    capa = doc.new_page()
    _escrever(capa, [
        "Justiça Federal da 1ª Região", "PJe - Processo Judicial Eletrônico", "",
        "04/11/2026" if quitado else "15/10/2026", "",
        f"Número: {NUMERO_DEMO}",
        "Classe: PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL",
        "Órgão julgador: 99ª Vara Federal de Juizado Especial Cível da SJGO",
        "Última distribuição : 10/03/2025",
        "Valor da causa: R$ 30.000,00",
        "Assuntos: Aposentadoria por Idade (DADOS FICTÍCIOS)", "",
        "Partes", f"{AUTOR} (AUTOR)", f"{ADVOGADO} (ADVOGADO)", "INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS (REU)", "",
        "Documentos", *linha_do_tempo,
    ])

    # --- peças das partes ---
    _pagina_documento(doc, "1000000001", "Petição inicial", ADVOGADO, "10/03/2025 11:04:01", [
        "EXCELENTÍSSIMO SENHOR JUIZ FEDERAL DO JUIZADO ESPECIAL FEDERAL DA SEÇÃO JUDICIÁRIA DE GOIÁS", "",
        f"{AUTOR}, brasileira, residente em Goiânia/GO, por seu advogado (OAB/GO nº 12345), vem propor a presente "
        "ação contra o INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS, pelos fatos e fundamentos a seguir.",
        "Requer a concessão da aposentadoria por idade, com pagamento das parcelas atrasadas desde a DER.",
        f"Nestes termos, pede deferimento. Goiânia, 10 de março de 2025. {ADVOGADO} - OAB/GO nº 12345",
    ])

    # --- uma página escaneada: só imagem, nenhum texto ---
    escaneada = doc.new_page()
    escaneada.draw_rect(pymupdf.Rect(60, 120, 535, 420), color=(0.5, 0.5, 0.5), fill=(0.85, 0.85, 0.85))

    # --- peças do juízo ---
    _pagina_documento(doc, "1000000003", "Sentença Tipo A", JUIZ, "20/08/2025 18:55:31", [
        "SENTENÇA", "",
        "Trata-se de ação previdenciária. Ante o exposto, JULGO PROCEDENTES os pedidos, nos termos do artigo 487, "
        "inciso I, do CPC, para condenar o INSS à concessão do benefício e ao pagamento das diferenças devidas desde a DER.",
        "Sem custas e tampouco honorários advocatícios (art. 55 da Lei 9.099/95).",
    ])
    _pagina_documento(doc, "1000000004", "Certidão de Trânsito em Julgado", SERVIDOR, "20/09/2025 09:10:00", [
        "CERTIDÃO", "", "Certifico o trânsito em julgado da sentença. Goiânia, 20 de setembro de 2025.",
    ])
    _pagina_documento(doc, "1000000005", "Decisão", JUIZ, "05/10/2026 14:20:10", [
        "DECISÃO", "",
        "Apresentados os cálculos pela parte autora, sem impugnação do INSS, HOMOLOGO os cálculos de fls. anteriores. "
        "Expeça-se ofício requisitório. "
        "Após a expedição da RPV, as partes serão intimadas do seu teor. "
        "Após o pagamento, a parte exequente será intimada para o levantamento dos valores, diretamente na agência "
        "bancária, sem expedição de alvará. Aguarde-se sua quitação.",
    ])
    _pagina_documento(doc, "1000000006",
                      "Ofício Requisitório de RPV - Requisição de Pequeno Valor (Requisição de Pagamento 2026.0000.099.000001)",
                      "SECRETARIA EXEMPLO", "12/10/2026 07:44:32", [
        "OFÍCIO REQUISITÓRIO DE PEQUENO VALOR (RPV)", "",
        f"Beneficiário: {AUTOR}",
        "Valor Total do Beneficiário: R$ 24.500,00",
        "Total de Valores de Exercícios Anteriores: R$ 24.500,00",
    ])

    # --- uma alegação da PARTE (não vale como prova) ---
    _pagina_documento(doc, "1000000007", "Manifestação", ADVOGADO, "14/10/2026 10:00:00", [
        "A parte autora entende que houve a satisfação da obrigação (art. 924, II, do CPC) e pede a extinção da execução.",
        f"{ADVOGADO} - OAB/GO nº 12345",
    ])

    if quitado:
        _pagina_documento(doc, "1000000008", "Certidão", SERVIDOR, "20/10/2026 16:00:00", [
            "CERTIDÃO", "", "Certifico que os valores foram depositados em conta à disposição da parte autora.",
        ])
        _pagina_documento(doc, "1000000009", "Decisão", JUIZ, "28/10/2026 11:30:00", [
            "DECISÃO", "",
            "Comprovado o levantamento dos valores pela parte autora, e não havendo outras pendências, JULGO EXTINTA a "
            "execução pelo pagamento, nos termos do art. 924, II, do CPC. Arquivem-se os autos.",
        ])

    doc.save(destino)
    doc.close()
    return Path(destino)


PASTA_DEMO = Path(tempfile.mkdtemp(prefix="demo_ribeiro_"))
SAIDA_DEMO = PASTA_DEMO / "output"
PDF_A = criar_pdf_demo(PASTA_DEMO / "processo_A_aguardando_pagamento.pdf", quitado=False)
PDF_B = criar_pdf_demo(PASTA_DEMO / "processo_B_pago.pdf", quitado=True)

for pdf in (PDF_A, PDF_B):
    with pymupdf.open(pdf) as d:
        print(f"{pdf.name}: {d.page_count} páginas")

processo_A_aguardando_pagamento.pdf: 8 páginas
processo_B_pago.pdf: 10 páginas


---
## 4. Passeio arquivo por arquivo

### 4.1 `app/constantes.py`: as regras de negócio

É aqui que mora a "inteligência" do MVP. Primeiro, **as fases do crédito** (a escada), em ordem:

In [7]:
pd.DataFrame(FASES, columns=["código da fase", "descrição"]).rename_axis("ordem")

,código da fase,descrição
ordem,,
0,SEM_INFORMACAO,Nenhuma informação de fase encontrada
1,SENTENCA,Sentença/condenação proferida
2,TRANSITO_EM_JULGADO,Transitou em julgado
3,CUMPRIMENTO_DE_SENTENCA,Em cumprimento de sentença / cálculos
4,REQUISICAO,Requisição (RPV/OPV/precatório) determinada ou expedida - aguardando pagamento
5,PAGAMENTO_DISPONIBILIZADO,Valor depositado/disponibilizado - aguardando levantamento
6,QUITADO,Dívida paga/liquidada (levantamento ou extinção por pagamento)


Depois, o **dicionário de termos**: para cada fase, os termos que indicam que o processo chegou ali (cada termo é um regex). Quantos termos há por fase:

In [8]:
pd.DataFrame([(fase, len(termos)) for fase, termos in DICIONARIO_TERMOS.items()], columns=["fase", "nº de termos"])

,fase,nº de termos
0,SENTENCA,3
1,TRANSITO_EM_JULGADO,3
2,CUMPRIMENTO_DE_SENTENCA,4
3,REQUISICAO,5
4,PAGAMENTO_DISPONIBILIZADO,6
5,QUITADO,8
6,ALERTA,1


Os termos da fase mais importante, **QUITADO** (a dívida foi paga):

In [9]:
pd.DataFrame(list(DICIONARIO_TERMOS["QUITADO"].items()), columns=["termo", "regex"])

,termo,regex
0,extincao_da_execucao,(?:extin[çc][ãa]o|extint[ao]s?|extingo)\s+(?:d[ao]s?\s+|a\s+|o\s+)?(?:execu[çc][ãa]o|cumprimento\s+de\s+senten[çc]a|obriga[çc][ãa]o)
1,art_924_inciso_II,"art(?:igo|\.)?\s*924,?\s*(?:inciso\s+)?II\b"
2,satisfacao_da_obrigacao,satisfa[çc][ãa]o\s+(?:integral\s+)?(?:d[ao]\s+)?(?:obriga[çc][ãa]o|cr[ée]dito|d[ée]bito|d[íi]vida)
3,levantamento_realizado,levantamento\s+(?:dos\s+valores\s+)?(?:foi\s+)?(?:efetuad\w+|realizad\w+|conclu[íi]d\w+|comprovad\w+)
4,valores_levantados,(?:valor(?:es)?|quantia|alvar[áa])\s+(?:j[áa]\s+)?(?:foi\s+|foram\s+)?levantad\w+
5,alvara_cumprido,alvar[áa]\s+(?:j[áa]\s+)?(?:foi\s+)?(?:cumprid\w+|liquidad\w+|pago)
6,quitacao_integral,quita[çc][ãa]o\s+(?:integral|total|plena)|d(?:ou|[áa])\s+(?:plena\s+|total\s+|integral\s+)?quita[çc][ãa]o
7,divida_paga,d[íi]vida\s+(?:foi\s+)?(?:paga|quitada|liquidada)


Aplicando o dicionário a uma frase (cada trecho encontrado ganha o nome do termo):

In [10]:
destacar("Comprovado o levantamento, JULGO EXTINTA a execução pelo pagamento, nos termos do art. 924, II, do CPC.",
         DICIONARIO_TERMOS["QUITADO"])
destacar("Assim, expeça-se ofício requisitório. Aguarde-se sua quitação.",
         DICIONARIO_TERMOS["REQUISICAO"])

> Repare no segundo exemplo: *"Aguarde-se sua **quitação**"* **não** foi marcado como pago. Palavras soltas como "quitação" ou "pagamento" **não** estão no dicionário (nos autos do INSS, "pagamento" aparece dezenas de vezes falando do *benefício*). Só entram **frases específicas**.

O arquivo também guarda os filtros de segurança (que veremos na seção 7) e os regex de metadados:

In [11]:
print("Palavras que tornam uma menção HIPOTÉTICA (futuro/condição):")
print(textwrap.fill(const.CONTEXTO_HIPOTETICO, 110))
print()
print("Modelo da linha de uma parte na capa do PJe:  NOME EM MAIÚSCULAS (AUTOR)")
print(textwrap.fill(const.REGEX_PARTE_COM_PAPEL, 110))

Palavras que tornam uma menção HIPOTÉTICA (futuro/condição):
\b(?:ap[óo]s|ser[áa]o?|caso|sob\s+pena|quando|dever[áa]o?|poder[áa]o?|aguard\w+|requer\w*|solicit\w*|pleite\w+
|eventual\w*|at[ée]\s+que|antes\s+d[aeo]s?|condicionad\w+|mediante|fins\s+de)\b

Modelo da linha de uma parte na capa do PJe:  NOME EM MAIÚSCULAS (AUTOR)
([A-ZÁÀÂÃÉÊÍÓÔÕÚÇ][A-ZÁÀÂÃÉÊÍÓÔÕÚÇ0-9 .,'&/-
]+?)\s*\((AUTOR|R[EÉ]U|REQUERENTE|REQUERIDO|EXEQUENTE|EXECUTAD[OA]|IMPETRANTE|IMPETRADO|APELANTE|APELADO|EMBAR
GANTE|EMBARGADO|ADVOGADO|ADVOGADA|PROCURADOR|PROCURADORA)\)


### 4.2 `app/erros.py`: erros próprios

Uma classe-base (`ErroRibeiro`) permite que o `main.py` capture **qualquer** erro do projeto e siga para o próximo PDF sem derrubar o lote.

In [12]:
mostrar_arquivo("app/erros.py")

```python
"""Modulo que implementa todas as classes de erro!"""


class ErroRibeiro(Exception):
    """Base de todos os erros do projeto (permite um único `except ErroRibeiro` no main)."""
    def __init__(self, mensagem):
        self.__msg = mensagem

    def __str__(self):
        return f"{type(self).__name__} ({self.__msg})"


# Leitura de PDF
class ErroPdfNaoEncontrado(ErroRibeiro):
    """O caminho informado não existe ou não é um arquivo."""


class ErroPdfProtegido(ErroRibeiro):
    """O PDF está protegido por senha."""


class ErroLeituraPdf(ErroRibeiro):
    """O arquivo não pôde ser aberto como PDF (corrompido, formato inválido...)."""

```

In [13]:
try:
    ler_pdf("arquivo_que_nao_existe.pdf")
except ErroRibeiro as erro:
    print(erro)

ErroPdfNaoEncontrado (Arquivo não encontrado: arquivo_que_nao_existe.pdf)


### 4.3 `app/classes.py`: os dados que circulam no pipeline

Três *dataclasses*: **`Pagina`** (texto de uma página), **`Evidencia`** (a "prova": um trecho que casou com um termo) e **`ResultadoProcesso`** (a resposta final).

In [14]:
mostrar_codigo(Pagina, Evidencia)

```python
@dataclass
class Pagina:
    """Uma página do PDF já convertida em texto."""
    numero: int                              # 1 = primeira página
    texto: str
    documento_id: Optional[str] = None       # PJe: id do documento a que a página pertence
    tipo_documento: Optional[str] = None     # PJe: "Sentença Tipo A", "Petição inicial"...
    assinante: Optional[str] = None          # PJe: quem assinou o documento

    @property
    def sem_texto(self) -> bool:
        return len(self.texto.strip()) < const.MIN_CARACTERES_PAGINA


@dataclass
class Evidencia:
    """Um trecho do PDF que casou com um termo do dicionário. É a 'prova' mostrada ao advogado."""
    fase: str                                # chave do DICIONARIO_TERMOS (ex.: "REQUISICAO")
    termo: str                               # nome do termo (ex.: "oficio_requisitorio")
    pagina: int
    trecho: str
    tipo_documento: Optional[str] = None
    hipotetico: bool = False                 # frase condicional/futura/negada ("aguarde-se o pagamento")
    origem_confiavel: bool = True            # False = veio de peça da parte (petição, manifestação...)

    @property
    def conta(self) -> bool:
        """Só evidências não hipotéticas e vindas do juízo decidem a fase."""
        return not self.hipotetico and self.origem_confiavel

```

A propriedade `conta` de `Evidencia` é o coração da confiabilidade: **só decide a fase** quem não é hipotética e vem de uma peça do juízo.

In [15]:
base = dict(fase="QUITADO", termo="extincao_da_execucao", pagina=8, trecho="...JULGO EXTINTA a execução...")
print("Evidência normal             -> conta?", Evidencia(**base).conta)
print("Evidência hipotética         -> conta?", Evidencia(**base, hipotetico=True).conta)
print("Evidência de peça da parte   -> conta?", Evidencia(**base, origem_confiavel=False).conta)

Evidência normal             -> conta? True
Evidência hipotética         -> conta? False
Evidência de peça da parte   -> conta? False


In [16]:
mostrar_codigo(ResultadoProcesso)

```python
@dataclass
class ResultadoProcesso:
    """Tudo o que o sistema descobriu sobre um PDF."""
    arquivo: str
    total_paginas: int = 0
    paginas_sem_texto: int = 0
    numero_processo: Optional[str] = None
    tribunal: Optional[str] = None
    classe: Optional[str] = None
    orgao_julgador: Optional[str] = None
    valor_causa: Optional[str] = None
    valor_requisitado: Optional[str] = None
    data_emissao_pdf: Optional[str] = None   # "posição" do processo quando o PDF foi gerado
    partes: list = field(default_factory=list)       # [{"nome": ..., "papel": ...}]
    advogados: list = field(default_factory=list)    # nomes
    oabs: list = field(default_factory=list)         # ["OAB/GO 12345", ...] mais frequentes
    juiz: Optional[str] = None
    fase: str = "SEM_INFORMACAO"
    fase_descricao: str = const.DESCRICAO_FASE["SEM_INFORMACAO"]
    divida_paga: str = const.DIVIDA_PAGA_SEM_INFO
    avisos: list = field(default_factory=list)
    evidencias: list = field(default_factory=list)   # list[Evidencia]

    def evidencia_principal(self) -> Optional[Evidencia]:
        """A evidência (que conta) da fase final: o que o advogado deve conferir primeiro."""
        for ev in self.evidencias:
            if ev.conta and ev.fase == self.fase:
                return ev
        return None

    def para_dict(self) -> dict:
        return asdict(self)

```

### 4.4 `app/funcoes.py`: o pipeline

Mapa das funções (a primeira linha da documentação de cada uma). Nas próximas seções vamos executar cada uma delas.

In [17]:
etapas = ["ler_pdf", "identificar_documentos", "extrair_metadados", "buscar_termos",
          "classificar_fase", "responder_divida_paga", "processar_pdf", "exportar_json", "exportar_csv"]
pd.DataFrame([(n, (inspect.getdoc(getattr(funcoes, n)) or "").splitlines()[0]) for n in etapas],
             columns=["função", "o que faz"])

,função,o que faz
0,ler_pdf,"Lê o PDF e devolve uma lista de Pagina (texto por página, na ordem natural de leitura)."
1,identificar_documentos,"PJe: descobre a qual documento (id, tipo, assinante) cada página pertence."
2,extrair_metadados,Extrai com regex os dados de identificação do processo. Campos não achados ficam fora do dict.
3,buscar_termos,Procura todos os termos do DICIONARIO_TERMOS e devolve Evidencias (página + trecho).
4,classificar_fase,A fase é a mais avançada da escada que tenha ao menos uma evidência que 'conta'.
5,responder_divida_paga,Traduz a fase na resposta direta: SIM / PROVAVEL / NAO / SEM_INFORMACAO.
6,processar_pdf,"Roda o pipeline completo em um PDF (ler, extrair, buscar, classificar) e devolve o ResultadoProcesso."
7,exportar_json,Um JSON completo (com todas as evidências) por PDF: output/<nome-do-pdf>.json
8,exportar_csv,Uma linha por processo em output/resumo.csv (separador ';' e BOM: abre direto no Excel pt-BR).


---
## 5. O pipeline passo a passo: do PDF ao JSON

Vamos seguir o **Processo A** (RPV expedida, ainda não paga).

### Passo 1: `ler_pdf`
Abre o PDF com PyMuPDF e devolve uma `Pagina` por página, na ordem natural de leitura. Linhas **rotacionadas** (como a assinatura vertical da margem do e-SAJ) são descartadas para não "sujar" o texto.

In [18]:
mostrar_codigo(ler_pdf, funcoes._texto_da_pagina)

```python
def ler_pdf(caminho) -> list:
    """Lê o PDF e devolve uma lista de Pagina (texto por página, na ordem natural de leitura).

    Levanta ErroPdfNaoEncontrado, ErroPdfProtegido ou ErroLeituraPdf.
    PDFs escaneados (só imagem) não dão erro: voltam com páginas `sem_texto`.
    """
    caminho = Path(caminho)
    if not caminho.is_file():
        raise ErroPdfNaoEncontrado(f"Arquivo não encontrado: {caminho}")
    try:
        documento = pymupdf.open(caminho)
    except Exception as erro:  # pymupdf lança vários tipos (FileDataError, EmptyFileError...)
        raise ErroLeituraPdf(f"Não foi possível abrir '{caminho.name}': {erro}") from erro

    with documento:
        if documento.needs_pass:
            raise ErroPdfProtegido(f"'{caminho.name}' está protegido por senha")
        paginas = [Pagina(numero=i, texto=_texto_da_pagina(pagina)) for i, pagina in enumerate(documento.pages(), start=1)]

    identificar_documentos(paginas)
    return paginas


def _texto_da_pagina(pagina) -> str:
    """Texto da página na ordem natural de leitura, SEM as linhas rotacionadas.

    O e-SAJ (TJSP) imprime a assinatura digital na vertical, na margem. Se entrar no texto,
    palavras da margem se misturam às da petição ("...DO ROBERTO ESTADO DE SÃO PAULO").
    `dir` é o vetor de direção da linha: (1, 0) = horizontal, esquerda para direita.
    sort=True ordena de cima para baixo / esquerda para direita.
    """
    linhas = []
    for bloco in pagina.get_text("dict", sort=True)["blocks"]:
        for linha in bloco.get("lines", []):  # blocos de imagem não têm "lines"
            if linha["dir"][0] >= 0.9:
                linhas.append("".join(span["text"] for span in linha["spans"]))
    return "\n".join(linhas)

```

In [19]:
paginas = ler_pdf(PDF_A)
print(f"{len(paginas)} páginas lidas. Texto da página 1 (a capa):\n")
print(paginas[0].texto)

8 páginas lidas. Texto da página 1 (a capa):

Justiça Federal da 1ª Região
PJe - Processo Judicial Eletrônico
15/10/2026
Número: 1234567-89.2025.4.01.3500
Classe: PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL
Órgão julgador: 99ª Vara Federal de Juizado Especial Cível da SJGO
Última distribuição : 10/03/2025
Valor da causa: R$ 30.000,00
Assuntos: Aposentadoria por Idade (DADOS FICTÍCIOS)
Partes
MARIA APARECIDA DE DEMONSTRACAO (AUTOR)
JOAO ADVOGADO EXEMPLO (ADVOGADO)
INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS (REU)
Documentos
10/03/2025  Petição inicial                      Polo ativo
10/03/2025  Carteira de identidade (escaneada)   Polo ativo
20/08/2025  Sentença Tipo A                      Interno
20/09/2025  Certidão de Trânsito em Julgado      Interno
05/10/2026  Decisão                              Interno
12/10/2026  Ofício Requisitório de RPV - Requisição de Pequeno Valor     Interno
14/10/2026  Manifestação                         Polo ativo


### Passo 2: `identificar_documentos`
Os autos do PJe juntam vários documentos num só PDF. Cada página traz um cabeçalho `Documento id … - tipo` e um rodapé `Assinado eletronicamente por … Num. … - Pág. N`. Com isso descobrimos **de que tipo é cada página e quem a assinou** (já foi chamado dentro do `ler_pdf`).

In [20]:
mostrar_codigo(identificar_documentos)

```python
def identificar_documentos(paginas: list) -> None:
    """PJe: descobre a qual documento (id, tipo, assinante) cada página pertence.

    Os autos do PJe juntam dezenas de documentos num só PDF. Cada página traz no rodapé
    "Assinado eletronicamente por: NOME - data ... Num. <id> - Pág. N" e o cabeçalho
    "Documento id <id> - <tipo>". Saber o TIPO permite (a) achar o juiz (quem assina a
    sentença) e (b) ignorar peças das partes na hora de decidir a fase.
    Em PDFs sem esse padrão (e-SAJ/TJSP) nada é preenchido, e tudo bem.
    """
    tipos = {}
    for pagina in paginas:
        for doc_id, tipo in const.RE_PJE_CABECALHO_DOC.findall(pagina.texto):
            tipos.setdefault(doc_id, normalizar_espacos(tipo))

    for pagina in paginas:
        achou = const.RE_PJE_NUM_DOC.search(pagina.texto)
        if not achou:
            continue
        pagina.documento_id = achou.group(1)
        pagina.tipo_documento = tipos.get(pagina.documento_id)
        assinatura = const.RE_PJE_ASSINANTE.search(pagina.texto)
        if assinatura:
            pagina.assinante = normalizar_espacos(assinatura.group(1))

```

In [21]:
pd.DataFrame([{
    "pág": p.numero, "documento_id": p.documento_id, "tipo_documento": p.tipo_documento,
    "assinante": p.assinante, "caracteres": len(p.texto.strip()), "sem_texto": p.sem_texto,
} for p in paginas])

,pág,documento_id,tipo_documento,assinante,caracteres,sem_texto
0,1,NaN,NaN,NaN,908,False
1,2,1000000001,Petição inicial,JOAO ADVOGADO EXEMPLO,654,False
2,3,NaN,NaN,NaN,0,True
3,4,1000000003,Sentença Tipo A,JOSE JUIZ FEDERAL EXEMPLO,446,False
4,5,1000000004,Certidão de Trânsito em Julgado,SERVIDOR EXEMPLO DA SILVA,247,False
5,6,1000000005,Decisão,JOSE JUIZ FEDERAL EXEMPLO,514,False
6,7,1000000006,Ofício Requisitório de RPV - Requisição de Pequeno Valor,SECRETARIA EXEMPLO,412,False
7,8,1000000007,Manifestação,JOAO ADVOGADO EXEMPLO,290,False


> A **página 3** é uma imagem escaneada: não tem texto. O sistema não "adivinha" nada nela, apenas **avisa** que não foi lida (veremos nos avisos). O tipo do documento será usado adiante para separar **peças do juízo** de **peças das partes**.

### Passo 3: `extrair_metadados`
Só regex: número CNJ, classe, órgão, valor da causa, partes, advogados, OABs e valor requisitado. O **juiz** é quem assina a sentença (no PJe) ou o `Juiz(a) de Direito:` (no e-SAJ).

In [22]:
mostrar_codigo(extrair_metadados, funcoes._extrair_juiz)

```python
def extrair_metadados(paginas: list) -> dict:
    """Extrai com regex os dados de identificação do processo. Campos não achados ficam fora do dict."""
    capa = "\n".join(p.texto for p in paginas[: const.PAGINAS_DA_CAPA])
    texto_total = "\n".join(p.texto for p in paginas)
    meta = {}

    for chave, regex in const.RE_CAPA_PJE.items():
        achou = regex.search(capa)
        if achou:
            meta[chave] = normalizar_espacos(achou.group(1))

    if "numero_processo" not in meta:  # sem capa estruturada: o número mais repetido é o do processo
        numeros = const.RE_CNJ.findall(texto_total)
        if numeros:
            meta["numero_processo"] = Counter(numeros).most_common(1)[0][0]

    if "classe" in meta:  # capa do PJe: a data no topo é a "posição" do processo
        achou = const.RE_DATA_EMISSAO_CAPA.search(paginas[0].texto)
        if achou:
            meta["data_emissao_pdf"] = achou.group(1)

    meta["partes"], meta["advogados"] = _extrair_partes_e_advogados(capa, texto_total)
    meta["oabs"] = _extrair_oabs(texto_total)

    juiz = _extrair_juiz(paginas, texto_total)
    if juiz:
        meta["juiz"] = juiz
    valor = _extrair_valor_requisitado(paginas)
    if valor:
        meta["valor_requisitado"] = valor
    if "valor_causa" in meta:
        meta["valor_causa"] = f"R$ {meta['valor_causa']}"
    return meta


def _extrair_juiz(paginas: list, texto_total: str):
    """e-SAJ: 'Juiz(a) de Direito: Dr(a). NOME'. PJe: quem assina a sentença (ou decisão/despacho)."""
    nomes_esaj = const.RE_JUIZ_ESAJ.findall(texto_total)
    if nomes_esaj:
        return Counter(nomes_esaj).most_common(1)[0][0]

    for tipo_procurado in (r"senten[çc]a", r"decis[ãa]o|despacho"):
        assinantes = [
            p.assinante for p in paginas
            if p.assinante and p.tipo_documento and re.search(tipo_procurado, p.tipo_documento, re.IGNORECASE)
        ]
        if assinantes:
            return Counter(assinantes).most_common(1)[0][0]
    return None

```

In [23]:
meta = extrair_metadados(paginas)
print(json.dumps(meta, ensure_ascii=False, indent=2))

{
  "numero_processo": "1234567-89.2025.4.01.3500",
  "classe": "PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL",
  "orgao_julgador": "99ª Vara Federal de Juizado Especial Cível da SJGO",
  "valor_causa": "R$ 30.000,00",
  "ultima_distribuicao": "10/03/2025",
  "assuntos": "Aposentadoria por Idade (DADOS FICTÍCIOS)",
  "data_emissao_pdf": "15/10/2026",
  "partes": [
    {
      "nome": "MARIA APARECIDA DE DEMONSTRACAO",
      "papel": "AUTOR"
    },
    {
      "nome": "INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS",
      "papel": "RÉU"
    }
  ],
  "advogados": [
    "JOAO ADVOGADO EXEMPLO"
  ],
  "oabs": [
    "OAB/GO 12345 (3x)"
  ],
  "juiz": "JOSE JUIZ FEDERAL EXEMPLO",
  "valor_requisitado": "R$ 24.500,00"
}


### Passo 4: `buscar_termos`
Aplica **todos** os termos do dicionário em **cada página** e guarda cada achado como uma `Evidencia` (página + trecho). Cada achado passa por dois testes:

- `_eh_hipotetico`: a frase tem palavras de futuro/condição (*após, será, aguarde, requer…*) ou uma negação colada no termo?
- `_origem_confiavel`: a página é de uma peça do **juízo** (sentença, decisão, certidão, ofício…) ou de uma **parte** (petição, manifestação)?

In [24]:
mostrar_codigo(funcoes._origem_confiavel, funcoes._eh_hipotetico, buscar_termos)

```python
def _origem_confiavel(tipo_documento) -> bool:
    """Sem tipo (capa/e-SAJ) é aceito; com tipo, só documentos do juízo."""
    return tipo_documento is None or bool(const.RE_TIPO_DOC_DO_JUIZO.search(tipo_documento))


def _eh_hipotetico(texto: str, inicio: int) -> bool:
    """True se, na frase que antecede o termo, há palavra de condição/futuro, ou uma negação colada nele."""
    janela = texto[max(0, inicio - const.JANELA_CONTEXTO): inicio]
    frase_atual = const.RE_FRONTEIRA_DE_FRASE.split(janela)[-1]
    return bool(
        const.RE_CONTEXTO_HIPOTETICO.search(frase_atual)
        or const.RE_NEGACAO_PROXIMA.search(frase_atual[-const.JANELA_NEGACAO:])
    )


def buscar_termos(paginas: list) -> list:
    """Procura todos os termos do DICIONARIO_TERMOS e devolve Evidencias (página + trecho)."""
    evidencias = []
    guardadas = Counter()
    metade = const.TAMANHO_TRECHO // 2

    for pagina in paginas:
        texto = normalizar_espacos(pagina.texto)
        confiavel = _origem_confiavel(pagina.tipo_documento)
        for fase, termo, regex in const.TERMOS_COMPILADOS:
            for achado in regex.finditer(texto):
                hipotetico = _eh_hipotetico(texto, achado.start())
                chave = (termo, hipotetico, confiavel)
                if guardadas[chave] >= const.MAX_EVIDENCIAS_POR_TERMO:
                    continue
                guardadas[chave] += 1
                evidencias.append(Evidencia(
                    fase=fase,
                    termo=termo,
                    pagina=pagina.numero,
                    trecho=texto[max(0, achado.start() - metade): achado.end() + metade],
                    tipo_documento=pagina.tipo_documento,
                    hipotetico=hipotetico,
                    origem_confiavel=confiavel,
                ))
    return evidencias

```

In [25]:
evidencias = buscar_termos(paginas)
df = pd.DataFrame([{
    "pág": e.pagina, "fase": e.fase, "termo": e.termo, "hipotético": e.hipotetico,
    "origem confiável": e.origem_confiavel, "conta?": e.conta, "tipo do documento": e.tipo_documento,
    "trecho": e.trecho[:120],
} for e in evidencias]).sort_values(["pág", "fase"], ignore_index=True)

print(f"{len(df)} evidências encontradas: {df['conta?'].sum()} contam, "
      f"{df['hipotético'].sum()} são hipotéticas, {(~df['origem confiável']).sum()} vêm de peça da parte.")

15 evidências encontradas: 9 contam, 3 são hipotéticas, 3 vêm de peça da parte.


**① Evidências que CONTAM**, as únicas que decidem a fase:

In [26]:
df[df["conta?"]][["pág", "fase", "termo", "tipo do documento", "trecho"]]

,pág,fase,termo,tipo do documento,trecho
0,1,REQUISICAO,oficio_requisitorio,NaN,8/2025 Sentença Tipo A Interno 20/09/2025 Certidão de Trânsito em Julgado Interno 05/10/2026 Decisão Interno 12/10/2026
1,1,TRANSITO_EM_JULGADO,certidao_transito,NaN,icial Polo ativo 10/03/2025 Carteira de identidade (escaneada) Polo ativo 20/08/2025 Sentença Tipo A Interno 20/09/2025
2,4,SENTENCA,julgo_procedente,Sentença Tipo A,"Documento id 1000000003 - Sentença Tipo A SENTENÇA Trata-se de ação previdenciária. Ante o exposto, JULGO PROCEDENTES os"
3,4,SENTENCA,condenar_ente_publico,Sentença Tipo A,"de ação previdenciária. Ante o exposto, JULGO PROCEDENTES os pedidos, nos termos do artigo 487, inciso I, do CPC, para"
4,5,TRANSITO_EM_JULGADO,certifico_transito,Certidão de Trânsito em Julgado,"Documento id 1000000004 - Certidão de Trânsito em Julgado CERTIDÃO Certifico o trânsito em julgado da sentença. Goiânia,"
5,5,TRANSITO_EM_JULGADO,certidao_transito,Certidão de Trânsito em Julgado,"Documento id 1000000004 - Certidão de Trânsito em Julgado CERTIDÃO Certifico o trânsito em julgado da sentença. Goiânia,"
9,6,REQUISICAO,oficio_requisitorio,Decisão,"Apresentados os cálculos pela parte autora, sem impugnação do INSS, HOMOLOGO os cálculos de fls. anteriores. Expeça-se"
10,7,REQUISICAO,oficio_requisitorio,Ofício Requisitório de RPV - Requisição de Pequeno Valor,Documento id 1000000006 - Ofício Requisitório de RPV - Requisição de Pequeno Valor (Requisição de Pagamento 2026.0000.09
11,7,REQUISICAO,requisicao_pagamento_numerada,Ofício Requisitório de RPV - Requisição de Pequeno Valor,Documento id 1000000006 - Ofício Requisitório de RPV - Requisição de Pequeno Valor (Requisição de Pagamento 2026.0000.09


**② Ignoradas por serem HIPOTÉTICAS** (texto-padrão no futuro da decisão: *"Após o pagamento, … será intimada para o levantamento…"*):

In [27]:
df[df["hipotético"]][["pág", "fase", "termo", "trecho"]]

,pág,fase,termo,trecho
6,6,CUMPRIMENTO_DE_SENTENCA,homologa_calculos,"Documento id 1000000005 - Decisão DECISÃO Apresentados os cálculos pela parte autora, sem impugnação do INSS, HOMOLOGO o"
7,6,PAGAMENTO_DISPONIBILIZADO,alvara_expedido,"Após o pagamento, a parte exequente será intimada para o levantamento dos valores, diretamente na agência bancária, sem"
8,6,PAGAMENTO_DISPONIBILIZADO,intimado_para_levantamento,"requisitório. Após a expedição da RPV, as partes serão intimadas do seu teor. Após o pagamento, a parte exequente será"


**③ Ignoradas por virem de PEÇA DA PARTE** (a manifestação do advogado *alega* extinção da execução, mas isso **não é prova**; só o juiz pode extinguir):

In [28]:
df[~df["origem confiável"]][["pág", "fase", "termo", "tipo do documento", "trecho"]]

,pág,fase,termo,tipo do documento,trecho
12,8,QUITADO,extincao_da_execucao,Manifestação,"id 1000000007 - Manifestação A parte autora entende que houve a satisfação da obrigação (art. 924, II, do CPC) e pede a"
13,8,QUITADO,art_924_inciso_II,Manifestação,"Documento id 1000000007 - Manifestação A parte autora entende que houve a satisfação da obrigação (art. 924, II, do CPC)"
14,8,QUITADO,satisfacao_da_obrigacao,Manifestação,"Documento id 1000000007 - Manifestação A parte autora entende que houve a satisfação da obrigação (art. 924, II, do CPC)"


### Passo 5: `classificar_fase` e `responder_divida_paga`
A fase do processo é a **mais alta da escada que tenha ao menos uma evidência que conta**. Depois a fase é traduzida na resposta direta: `SIM` / `PROVAVEL` / `NAO` / `SEM_INFORMACAO`.

In [29]:
mostrar_codigo(classificar_fase, responder_divida_paga)

```python
def classificar_fase(evidencias: list) -> str:
    """A fase é a mais avançada da escada que tenha ao menos uma evidência que 'conta'."""
    fase = "SEM_INFORMACAO"
    for ev in evidencias:
        if ev.conta and ev.fase in const.ORDEM_FASE and const.ORDEM_FASE[ev.fase] > const.ORDEM_FASE[fase]:
            fase = ev.fase
    return fase


def responder_divida_paga(fase: str) -> str:
    """Traduz a fase na resposta direta: SIM / PROVAVEL / NAO / SEM_INFORMACAO."""
    if fase == "QUITADO":
        return const.DIVIDA_PAGA_SIM
    if fase == "PAGAMENTO_DISPONIBILIZADO":
        return const.DIVIDA_PAGA_PROVAVEL
    if fase == "SEM_INFORMACAO":
        return const.DIVIDA_PAGA_SEM_INFO
    return const.DIVIDA_PAGA_NAO

```

In [30]:
fase = classificar_fase(evidencias)
resposta = responder_divida_paga(fase)
desenhar_escada(fase, f"<b>Fase:</b> {fase} &nbsp;|&nbsp; <b>Dívida paga?</b> {resposta}")

### Passo 6: `processar_pdf`
Junta os passos 1 a 5 e preenche o `ResultadoProcesso`, incluindo os **avisos** (páginas não lidas, menções condicionais ignoradas, sentença "improcedente").

In [31]:
mostrar_codigo(processar_pdf)

```python
def processar_pdf(caminho) -> ResultadoProcesso:
    """Roda o pipeline completo em um PDF (ler, extrair, buscar, classificar) e devolve o ResultadoProcesso."""
    caminho = Path(caminho)
    paginas = ler_pdf(caminho)
    sem_texto = sum(p.sem_texto for p in paginas)
    resultado = ResultadoProcesso(arquivo=caminho.name, total_paginas=len(paginas), paginas_sem_texto=sem_texto)

    if sem_texto:
        resultado.avisos.append(_aviso_paginas_sem_texto(paginas, sem_texto))

    meta = extrair_metadados(paginas)
    resultado.numero_processo = meta.get("numero_processo")
    resultado.tribunal = identificar_tribunal(resultado.numero_processo) if resultado.numero_processo else None
    resultado.classe = meta.get("classe")
    resultado.orgao_julgador = meta.get("orgao_julgador")
    resultado.valor_causa = meta.get("valor_causa")
    resultado.valor_requisitado = meta.get("valor_requisitado")
    resultado.data_emissao_pdf = meta.get("data_emissao_pdf")
    resultado.partes = meta["partes"]
    resultado.advogados = meta["advogados"]
    resultado.oabs = meta["oabs"]
    resultado.juiz = meta.get("juiz")

    if not resultado.numero_processo:
        resultado.avisos.append("Número do processo (CNJ) não encontrado.")

    resultado.evidencias = buscar_termos(paginas)
    resultado.fase = classificar_fase(resultado.evidencias)
    resultado.fase_descricao = const.DESCRICAO_FASE[resultado.fase]
    resultado.divida_paga = responder_divida_paga(resultado.fase)

    if any(ev.fase == "ALERTA" and ev.conta for ev in resultado.evidencias):
        resultado.avisos.append(
            "Encontrado 'julgo improcedente' em peça do juízo: confirme se existe mesmo crédito a receber."
        )
    resultado.avisos.extend(_avisar_mencoes_ignoradas(resultado))
    return resultado

```

In [32]:
resultado_a = processar_pdf(PDF_A)
print(formatar_resumo(resultado_a))

=== processo_A_aguardando_pagamento.pdf
  Processo : 1234567-89.2025.4.01.3500  (TRF1)  | 8 páginas
  Classe   : PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL  | Órgão: 99ª Vara Federal de Juizado Especial Cível da SJGO
  Partes   : MARIA APARECIDA DE DEMONSTRACAO (AUTOR); INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS (RÉU)
  Advogados: JOAO ADVOGADO EXEMPLO
  OAB      : OAB/GO 12345 (3x)
  Juiz     : JOSE JUIZ FEDERAL EXEMPLO
  Valores  : causa R$ 30.000,00 | requisitado R$ 24.500,00
  FASE     : REQUISICAO - Requisição (RPV/OPV/precatório) determinada ou expedida - aguardando pagamento
  DÍVIDA PAGA? NAO   (PDF gerado em 15/10/2026)
  [!] 1/8 páginas sem texto (imagem/scan) NÃO foram lidas: pág. 3. Rode OCR para lê-las (veja README).
  [!] Menções condicionais/futuras a pagamento ou quitação foram ignoradas (pág. 6). Se alguma for pagamento real, a fase está subestimada.
  > pág. 1 [oficio_requisitorio]: ...8/2025 Sentença Tipo A Interno 20/09/2025 Certidão de Trânsito em Julgado Interno 05/10

Repare nos dois avisos `[!]`. Eles existem porque **errar para "não pago" é o erro caro** (comprar um crédito que já foi quitado), então nada é descartado em silêncio: o sistema diz *onde* um humano deve olhar.

### Passo 7: `exportar_json` 🎯
O resultado completo, **com todas as evidências** (inclusive as ignoradas, marcadas), vira um arquivo JSON.

In [33]:
mostrar_codigo(exportar_json)

```python
def exportar_json(resultado: ResultadoProcesso, pasta) -> Path:
    """Um JSON completo (com todas as evidências) por PDF: output/<nome-do-pdf>.json"""
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    destino = pasta / f"{Path(resultado.arquivo).stem}.json"
    destino.write_text(json.dumps(resultado.para_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
    return destino

```

In [34]:
arquivo_json = exportar_json(resultado_a, SAIDA_DEMO)
linhas = arquivo_json.read_text(encoding="utf-8").splitlines()
print(f"Arquivo gerado: {arquivo_json.name}  ({len(linhas)} linhas)\n")
print("\n".join(linhas[:42]))
print("   ...")

Arquivo gerado: processo_A_aguardando_pagamento.json  (173 linhas)

{
  "arquivo": "processo_A_aguardando_pagamento.pdf",
  "total_paginas": 8,
  "paginas_sem_texto": 1,
  "numero_processo": "1234567-89.2025.4.01.3500",
  "tribunal": "TRF1",
  "classe": "PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL",
  "orgao_julgador": "99ª Vara Federal de Juizado Especial Cível da SJGO",
  "valor_causa": "R$ 30.000,00",
  "valor_requisitado": "R$ 24.500,00",
  "data_emissao_pdf": "15/10/2026",
  "partes": [
    {
      "nome": "MARIA APARECIDA DE DEMONSTRACAO",
      "papel": "AUTOR"
    },
    {
      "nome": "INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS",
      "papel": "RÉU"
    }
  ],
  "advogados": [
    "JOAO ADVOGADO EXEMPLO"
  ],
  "oabs": [
    "OAB/GO 12345 (3x)"
  ],
  "juiz": "JOSE JUIZ FEDERAL EXEMPLO",
  "fase": "REQUISICAO",
  "fase_descricao": "Requisição (RPV/OPV/precatório) determinada ou expedida - aguardando pagamento",
  "divida_paga": "NAO",
  "avisos": [
    "1/8 páginas sem texto (ima

No mesmo arquivo, logo abaixo, vêm as evidências. Cada uma é a **prova** que o advogado confere no PDF (página + trecho), aqui a que **decidiu a fase**:

In [35]:
dados = json.loads(arquivo_json.read_text(encoding="utf-8"))
decisiva = next(e for e in dados["evidencias"] if e["fase"] == dados["fase"] and not e["hipotetico"] and e["origem_confiavel"])
print(json.dumps(decisiva, ensure_ascii=False, indent=2))

{
  "fase": "REQUISICAO",
  "termo": "oficio_requisitorio",
  "pagina": 1,
  "trecho": "8/2025 Sentença Tipo A Interno 20/09/2025 Certidão de Trânsito em Julgado Interno 05/10/2026 Decisão Interno 12/10/2026 Ofício Requisitório de RPV - Requisição de Pequeno Valor Interno 14/10/2026 Manifestação Polo ativo",
  "tipo_documento": null,
  "hipotetico": false,
  "origem_confiavel": true
}


---
## 6. `main.py`: tudo junto, pela linha de comando

O `main.py` percorre os PDFs, mostra o resumo no terminal e grava um JSON por processo e um `resumo.csv`. Se um PDF falhar, o erro é mostrado e o lote **continua**.

In [36]:
mostrar_arquivo("main.py")

```python
"""Ponto de entrada do MVP.

Uso:
    python main.py                       # processa todos os PDFs de input/
    python main.py input/processo.pdf    # processa um PDF (ou vários)
    python main.py --saida resultados    # grava em outra pasta (padrão: output/)
"""

import argparse
import sys
import time
from pathlib import Path

from app import (
    PASTA_INPUT, PASTA_OUTPUT, ErroRibeiro,
    exportar_csv, exportar_json, formatar_resumo, listar_pdfs, processar_pdf,
)


def ler_argumentos():
    parser = argparse.ArgumentParser(description="Extrai dados de PDFs de processos e diz se a dívida foi paga.")
    parser.add_argument("pdfs", nargs="*", type=Path, help=f"PDFs a processar (padrão: todos de {PASTA_INPUT.name}/)")
    parser.add_argument("--saida", type=Path, default=PASTA_OUTPUT, help=f"pasta de saída (padrão: {PASTA_OUTPUT.name}/)")
    return parser.parse_args()


def main() -> int:
    args = ler_argumentos()
    pdfs = args.pdfs or listar_pdfs(PASTA_INPUT)
    if not pdfs:
        print(f"Nenhum PDF encontrado em {PASTA_INPUT}/. Coloque os arquivos lá ou passe o caminho.")
        return 1

    resultados, falhas = [], 0
    for pdf in pdfs:
        inicio = time.time()
        try:
            resultado = processar_pdf(pdf)
        except ErroRibeiro as erro:  # um PDF ruim não derruba o lote
            print(f"[ERRO] {pdf.name}: {erro}\n")
            falhas += 1
            continue
        exportar_json(resultado, args.saida)
        resultados.append(resultado)
        print(formatar_resumo(resultado))
        print(f"  ({time.time() - inicio:.1f}s)\n")

    if resultados:
        csv_gerado = exportar_csv(resultados, args.saida)
        print(f"{len(resultados)} processo(s) analisado(s), {falhas} falha(s). Resumo: {csv_gerado}")
    return 1 if falhas and not resultados else 0


if __name__ == "__main__":
    sys.exit(main())

```

Rodando como o usuário rodaria (`python main.py <pdfs> --saida <pasta>`), agora com os **dois** PDFs. Repare como, semanas depois, o **Processo B** chega em `QUITADO`:

In [37]:
proc = subprocess.run(
    [sys.executable, "main.py", str(PDF_A), str(PDF_B), "--saida", str(SAIDA_DEMO)],
    capture_output=True, text=True, cwd=RAIZ_PROJETO,
)
print(proc.stdout)
if proc.returncode:
    print("ERRO:", proc.stderr)

=== processo_A_aguardando_pagamento.pdf
  Processo : 1234567-89.2025.4.01.3500  (TRF1)  | 8 páginas
  Classe   : PROCEDIMENTO DO JUIZADO ESPECIAL CÍVEL  | Órgão: 99ª Vara Federal de Juizado Especial Cível da SJGO
  Partes   : MARIA APARECIDA DE DEMONSTRACAO (AUTOR); INSTITUTO NACIONAL DO SEGURO SOCIAL - INSS (RÉU)
  Advogados: JOAO ADVOGADO EXEMPLO
  OAB      : OAB/GO 12345 (3x)
  Juiz     : JOSE JUIZ FEDERAL EXEMPLO
  Valores  : causa R$ 30.000,00 | requisitado R$ 24.500,00
  FASE     : REQUISICAO - Requisição (RPV/OPV/precatório) determinada ou expedida - aguardando pagamento
  DÍVIDA PAGA? NAO   (PDF gerado em 15/10/2026)
  [!] 1/8 páginas sem texto (imagem/scan) NÃO foram lidas: pág. 3. Rode OCR para lê-las (veja README).
  [!] Menções condicionais/futuras a pagamento ou quitação foram ignoradas (pág. 6). Se alguma for pagamento real, a fase está subestimada.
  > pág. 1 [oficio_requisitorio]: ...8/2025 Sentença Tipo A Interno 20/09/2025 Certidão de Trânsito em Julgado Interno 05/10

In [38]:
resultado_b = processar_pdf(PDF_B)
desenhar_escada(resultado_a.fase, f"<b>Processo A</b> (PDF de {resultado_a.data_emissao_pdf}): {resultado_a.fase}, dívida paga? <b>{resultado_a.divida_paga}</b>")
desenhar_escada(resultado_b.fase, f"<b>Processo B</b> (PDF de {resultado_b.data_emissao_pdf}): {resultado_b.fase}, dívida paga? <b>{resultado_b.divida_paga}</b>")

O `resumo.csv` (uma linha por processo; abre direto no Excel) e os arquivos gerados:

In [39]:
resumo = pd.read_csv(SAIDA_DEMO / "resumo.csv", sep=";", encoding="utf-8-sig")
display(resumo[["arquivo", "numero_processo", "tribunal", "juiz", "valor_requisitado", "fase", "divida_paga", "pagina_evidencia"]])
print("Arquivos em output/:", sorted(p.name for p in SAIDA_DEMO.iterdir()))

,arquivo,numero_processo,tribunal,juiz,valor_requisitado,fase,divida_paga,pagina_evidencia
0,processo_A_aguardando_pagamento.pdf,1234567-89.2025.4.01.3500,TRF1,JOSE JUIZ FEDERAL EXEMPLO,"R$ 24.500,00",REQUISICAO,NAO,1
1,processo_B_pago.pdf,1234567-89.2025.4.01.3500,TRF1,JOSE JUIZ FEDERAL EXEMPLO,"R$ 24.500,00",QUITADO,SIM,10


Arquivos em output/: ['processo_A_aguardando_pagamento.json', 'processo_B_pago.json', 'resumo.csv']


---
## 7. Por que confiar no resultado

Autos judiciais estão cheios de texto no futuro, alegações das partes e negações. Os filtros da seção 4 existem por causa de casos **reais** que apareceram nos testes. Primeiro, uma bateria de frases, com o resultado **esperado** e o **obtido**:

In [40]:
def fase_de(texto, tipo=None):
    return classificar_fase(buscar_termos([Pagina(numero=1, texto=texto, tipo_documento=tipo)]))

casos = [
    ("Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.", None, "QUITADO"),
    ("Não havendo outras pendências, julgo extinta a execução.", None, "QUITADO"),
    ("Assim, expeça-se ofício requisitório. Aguarde-se sua quitação.", None, "REQUISICAO"),
    ("Após o pagamento, a parte será intimada para o levantamento dos valores.", None, "SEM_INFORMACAO"),
    ("Informa-se que não houve pagamento efetuado pelo ente devedor.", None, "SEM_INFORMACAO"),
    ("Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.", "Manifestação", "SEM_INFORMACAO"),
    ("Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.", "Sentença Tipo A", "QUITADO"),
    ("Aplica-se o CPC no que não conflitar com a Lei 10.259, certifico o trânsito em julgado.", None, "TRANSITO_EM_JULGADO"),
]
linhas = []
for texto, tipo, esperado in casos:
    obtido = fase_de(texto, tipo)
    linhas.append({"texto": texto, "tipo do documento": tipo or "(juízo / sem tipo)",
                   "esperado": esperado, "obtido": obtido, "": "✅" if obtido == esperado else "❌"})
pd.DataFrame(linhas)

,texto,tipo do documento,esperado,obtido,
0,"Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.",(juízo / sem tipo),QUITADO,QUITADO,✅
1,"Não havendo outras pendências, julgo extinta a execução.",(juízo / sem tipo),QUITADO,QUITADO,✅
2,"Assim, expeça-se ofício requisitório. Aguarde-se sua quitação.",(juízo / sem tipo),REQUISICAO,REQUISICAO,✅
3,"Após o pagamento, a parte será intimada para o levantamento dos valores.",(juízo / sem tipo),SEM_INFORMACAO,SEM_INFORMACAO,✅
4,Informa-se que não houve pagamento efetuado pelo ente devedor.,(juízo / sem tipo),SEM_INFORMACAO,SEM_INFORMACAO,✅
5,"Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.",Manifestação,SEM_INFORMACAO,SEM_INFORMACAO,✅
6,"Julgo extinta a execução pelo pagamento, art. 924, II, do CPC.",Sentença Tipo A,QUITADO,QUITADO,✅
7,"Aplica-se o CPC no que não conflitar com a Lei 10.259, certifico o trânsito em julgado.",(juízo / sem tipo),TRANSITO_EM_JULGADO,TRANSITO_EM_JULGADO,✅


**E se os filtros não existissem?** Vamos desligar cada um, de propósito, e reprocessar o Processo A (que **não** está pago):

In [41]:
def resultado_do_processo_a(titulo, **trocas):
    contexto = mock.patch.multiple(funcoes, **trocas) if trocas else contextlib.nullcontext()
    with contexto:
        r = processar_pdf(PDF_A)
    return {"cenário": titulo, "fase": r.fase, "dívida paga?": r.divida_paga}

pd.DataFrame([
    resultado_do_processo_a("Sistema completo (como no projeto)"),
    resultado_do_processo_a("SEM o filtro de contexto hipotético", _eh_hipotetico=lambda *a, **k: False),
    resultado_do_processo_a("SEM o filtro de origem (aceita peça da parte)", _origem_confiavel=lambda *a, **k: True),
])

,cenário,fase,dívida paga?
0,Sistema completo (como no projeto),REQUISICAO,NAO
1,SEM o filtro de contexto hipotético,PAGAMENTO_DISPONIBILIZADO,PROVAVEL
2,SEM o filtro de origem (aceita peça da parte),QUITADO,SIM


Sem o filtro hipotético, o texto-padrão *"após o pagamento… será intimada para o levantamento"* faz o processo parecer com **pagamento disponibilizado**. Sem o filtro de origem, a **alegação do advogado** faz o processo parecer **quitado**. Os dois são falsos positivos; o primeiro foi observado em autos reais do TRF1.

**Garantia de qualidade**: 24 testes automatizados cobrem esses casos (rodam em menos de 1 segundo) e, ao desligar o filtro de propósito, os testes pertinentes **falham**, como deveriam:

```bash
python -m unittest discover -s tests -t . -v
```

---
## 8. Limitações e próximos passos

**Limitações (sem rodeios)**

- **Falta validar com processos realmente pagos.** Os 3 PDFs reais testados estão na fase *requisição expedida*, nenhum pago. O caminho `SIM`/`PROVAVEL` foi validado só com texto sintético (como o Processo B acima).
- **PDFs escaneados não são lidos** (OCR não implementado). Num dos processos reais, 52 de 72 páginas eram imagem; o sistema lista as páginas não lidas.
- **Diário da Justiça (DJe/DJEN) não foi testado**: os PDFs entregues são autos do processo.
- A fase é a *mais alta com evidência*: um único trecho mal interpretado distorce o resultado, por isso a resposta **sempre traz a página** para conferência.

**Próximos passos**

1. Receber 3 a 5 PDFs de processos **já pagos** e calibrar o dicionário.
2. OCR (Tesseract) apenas nas páginas sem texto.
3. Consulta por **número do processo**: avaliar a API pública do **DataJud (CNJ)** em vez de raspar sites de tribunais.
4. Calibrar a leitura de Diário da Justiça com um PDF real.
5. Só então, um LLM para campos ambíguos, mantendo regex como primeira camada e a evidência (página/trecho) como saída obrigatória.

---
### Apêndice: rodar com os PDFs reais *(opcional, cuidado com dados pessoais)*

Mude `USAR_PDFS_REAIS` para `True` para processar os arquivos de `input/`. ⚠️ **As saídas conterão nomes, OAB e trechos de clientes.** Antes de salvar ou versionar o notebook, use *Edit → Clear Outputs of All Cells*.

In [42]:
USAR_PDFS_REAIS = False

if USAR_PDFS_REAIS:
    for pdf in listar_pdfs(PASTA_INPUT):
        print(formatar_resumo(processar_pdf(pdf)), "\n")
else:
    print("Desligado: nenhum PDF real foi lido neste notebook.")

Desligado: nenhum PDF real foi lido neste notebook.
